# Local revision storage

Fetch one ALFRED revision history, save it to SQLite, close the database, and load the same history from disk. The local database is ignored by Git.

In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from IPython.display import display

from asof import SQLiteRevisionStore, get_revision_history

ENV_PATH = Path(find_dotenv())
PROJECT_ROOT = ENV_PATH.parent
DATABASE_PATH = PROJECT_ROOT / "data" / "asof.sqlite3"

load_dotenv(ENV_PATH)
API_KEY = os.getenv("FRED_API_KEY")
assert API_KEY, "FRED_API_KEY was not loaded from .env"

In [2]:
history = get_revision_history(
    series_id="PAYEMS",
    observation_date="2008-08-01",
    api_key=API_KEY,
)

with SQLiteRevisionStore(DATABASE_PATH) as store:
    rows_written = store.save_history(history)

print(f"Saved {rows_written} vintages to data/{DATABASE_PATH.name}")

Saved 16 vintages to data/asof.sqlite3


In [3]:
with SQLiteRevisionStore(DATABASE_PATH) as store:
    restored = store.load_history(
        series_id="PAYEMS",
        observation_date="2008-08-01",
    )
    stored_dates = store.list_observations(series_id="PAYEMS")

assert restored == history
print(f"Reloaded after reopening: {restored == history}")
print(f"Stored PAYEMS observations: {stored_dates}")

Reloaded after reopening: True
Stored PAYEMS observations: ('2008-08-01',)


In [4]:
stored_revisions = pd.DataFrame(
    [
        {"vintage_date": revision.vintage_date, "value_thousands": revision.value}
        for revision in restored.revisions
    ]
)
display(stored_revisions.style.format(precision=1, thousands=","))

,vintage_date,value_thousands
0,2008-09-05,"137,473.0"
1,2008-10-03,"137,477.0"
2,2008-11-07,"137,423.0"
3,2009-02-06,"137,053.0"
4,2010-02-05,"136,741.0"
5,2011-02-04,"136,747.0"
6,2012-02-03,"136,764.0"
7,2013-02-01,"136,836.0"
8,2014-02-07,"137,162.0"
9,2016-02-05,"137,233.0"
